In [ ]:
import pyinaturalist

In [ ]:
# pyinaturalist.get_observations(
#     taxon_id="48715",
    
# )

In [ ]:
import requests, random

species = {
    "Amanita phalloides": "deadly",
    "Cantharellus": "edible",
    "Amanita muscaria": "poisonous",
    "Boletus edulis": "edible"
}

name = random.choice(list(species))
r = requests.get("https://api.inaturalist.org/v1/observations", params={
    "taxon_name": "Amanita muscaria",
    "quality_grade": "research",
    "photos": "true", "photo_license": "cc-by,cc-by-nc,cc0",
    "per_page": 30, "order_by": "random"})
photo = random.choice(r.json()["results"])["photos"][0]["url"].replace("square", "large")
# print(name, species[name], photo)

In [ ]:
from bs4 import BeautifulSoup
with open("ukr_shroom_all.txt") as html_doc:
    allfile = html_doc.read()
    soup = BeautifulSoup(html_doc, 'html.parser')

In [ ]:
import re
from tqdm import tqdm

statuses = []
edible = []
cond = []
inedible = []
poisonous = []
# ukr_names = []
# lat_names = []
# img_links = {}
list_entries = allfile.split("<li")[1:]
for entry in tqdm(list_entries):
    
    status = re.findall(r'data-k="([^"]*)"', entry)[0]
    ukr_name = re.findall(r"<b>(.*?)</b>", entry, re.DOTALL)[0]
    lat_name = re.findall(r"<em>(.*?)</em>", entry, re.DOTALL)[0]
    
    try:
        r = requests.get("https://api.inaturalist.org/v1/observations", params={
                "taxon_name": lat_name,
                "quality_grade": "research",
                "photos": "true", "photo_license": "cc-by,cc-by-nc,cc0",
                "per_page": 30, "order_by": "random"})
    except:
        print(f"Skipped {lat_name}")
        continue
    img_links = [res["photos"][0]["url"].replace("square", "large") for res in r.json()["results"]]
    
    struct = {"ukr": ukr_name, "lat": lat_name, "links": img_links}
    if status == "ed":
        edible.append(struct)
    elif status == "um":
        cond.append(struct)
    elif status in ["ne"]:
        inedible.append(struct)
    elif status in ["ot"]:
        poisonous.append(struct)
    

In [ ]:
import pandas as pd

pd.DataFrame(edible).to_csv('edible.csv')
pd.DataFrame(cond).to_csv('cond.csv')
pd.DataFrame(inedible).to_csv('inedible.csv')
pd.DataFrame(poisonous).to_csv('poisonous.csv')

In [ ]:
import ast

pds = []
vals_score = {
    "cond": "edible",
    "edible": "edible",
    "inedible": "inedible",
    "poisonous": "poisonous"
}

def process_urls(url_list):
    l = ast.literal_eval(url_list)
    for i, s in enumerate(l):
        l[i] = s.replace("large", "medium")
    return l

for name in ["cond", "edible", "inedible", "poisonous"]:
    data = pd.read_json(f"{name}.json")
    data = data.rename(columns={"ukr": "common_name", "lat": "scientific_name", "links": "images"})
    data["edibility"] = vals_score[name]
    data["edibility_precise"] = name
    data["images"] = data["images"].apply(process_urls)
    pds.append(data)
    
pd.concat(pds, ignore_index=True).to_json("shrooms.json", index=False, orient="records", indent=3, force_ascii=False)